# Books Banned in Texas State Prisons

In [44]:
# step 1: import pandas
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
import string

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

# the 'magic' or the percent symbol means it will look good visually

In [45]:
# step 2: import .csv file
data = pd.read_csv("data/banned_book_data_tx-list.csv")
corpus = pd.read_csv("data/corpus.csv")
# Book bans limit access to information ad restrict freedom of expression. There has been no comprehensive data for training ML models on if a book will be censored (challenged/banned) or not. This dataset aims to address that. The title and author of banned books are obtained through non-profits like the ALA and Pen America while metadata like description and genre for them are obtained through webscraping Goodreads.
# The books that are labeled as uncensored are obtained through kaggle then filtered.

# output the first few data points
# data.head()
corpus.head(10)

# note on sourcing: got this information through a request to the tdcj public information office, received 10/14/2022

,Author,Title,Description,Genre,Banned
0,"Laurie, Forest",The Black Witch,A new Black Witch will rise…her powers vast be...,"Fantasy, Young Adult, Witches, Romance, Magic,...",1
1,"Deborah, Harkness",Shadow of Night,Picking up from A Discovery of Witches’ cliffh...,"Fantasy, Romance, Fiction, Historical Fiction,...",1
2,"Diana, Athill",Somewhere Towards the End,Winner of the 2009 National Book Critics Circl...,"Memoir, Nonfiction, Biography, Autobiography, ...",1
3,"Cassandra, Clare",Chain Of Iron,The Shadowhunters must catch a killer in Edwar...,"Fantasy, Young Adult, Romance, Historical Fict...",1
4,"Sonora, Reyes",The Lesbiana's Guide to Catholic School,A debut novel about a queer Mexican American g...,"Romance, LGBT, Young Adult, Queer, Lesbian, Co...",1
5,"Amy, Ignatow",The Rocky Road Trip of Lydia Goldblatt and Jul...,"It’s summertime, and Julie and Lydia are going...","Middle Grade, Graphic Novels, Childrens, Humor...",1
6,"Ian, McEwan",On Chesil Beach,A novel of remarkable depth and poignancy from...,"Fiction, Historical Fiction, Literary Fiction,...",1
7,"Suzanne, Brockmann",Over the Edge,Suzanne Brockmann has taken romantic suspense ...,"Romance, Romantic Suspense, Suspense, Military...",1
8,"Dav, Pilkey",Captain Underpants and the Sensational Saga of...,"Laugh out loud with Captain Underpants, the #1...","Childrens, Humor, Fiction, Graphic Novels, Mid...",1
9,"Mary Higgins, Clark",Moonlight Becomes You,"Newport, Rhode Island: a world of old money, o...","Mystery, Fiction, Suspense, Thriller, Mystery ...",1


## Inspect the Data

In [46]:
data.shape

(9396, 10)

In [47]:
data.columns

Index(['publication', 'author', 'date', 'unit_deny_reason', 'reason',
       'note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022.',
       'year', 'month', 'day', 'state_arc'],
      dtype='object')

In [48]:
corpus.shape

(17440, 5)

In [49]:
corpus.columns

Index(['Author', 'Title', 'Description', 'Genre', 'Banned'], dtype='object')

## Clean up Column Names

In [50]:
data = data.rename(columns={
    "publication": "publication",
    "author": "author",
    "date": "date",
    "unit_deny_reason": "unit_reason",
    "reason": "reason",
    "year": "year",
    "month": "month",
    "day": "day",
    "state_arc": "state"
})

data = data.drop(columns=["note_on_sourcing:_i_got_this_information_through_a_request_to_the_tdcj_public_information_office,_received_10/14/2022."], errors="ignore")
data.head()

,publication,author,date,unit_reason,reason,year,month,day,state
0,...AND THEIR MEMORY WAS A BITTER TREE,"HOWARD, ROBERT",2016-08-19,PAGES 81 & 369 SEXUALLY EXPLICIT IMAGE,PGS 81 & 369 SEXUALLY EXPLICIT IMAGE,2016,8,19,tx
1,"(NON)CONFORM RUSSIAN AND SOVIET ART 1958-1995,","THIEMANN, BARBARA",2012-09-10,PAGE 422 PHOTO OF A NUDE CHILD,PG 422 NUDE CHILD,2012,9,10,tx
2,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2019-07-17,PG 294 SEXUALLY EXPLICIT IMAGE,PG 294 SEXUALLY EXPLICIT IMAGE,2019,7,17,tx
3,SCUSE ME WHILE I KISS THE SKY,"HENDERSON, DAVID",2010-01-29,PAGE 310 CONTAINS SEXUALLY EXPLICIT IMAGES,PAGE 9 OF PHOTO INSERT CONTAINS SEXUALLY EXPLI...,2010,1,29,tx
4,A BLACK GAZE,"CAMPT, TINA",2021-09-08,"PAGES 28, 30, 35 SEXUALLY EXPLICIT IMAGES","PAGES 28, 30 & 35 CONTAIN SEXUALLY EXPLICIT IM...",2021,9,8,tx


## Reinspect the Data

In [51]:
data.shape

(9396, 9)

In [52]:
data.columns

Index(['publication', 'author', 'date', 'unit_reason', 'reason', 'year',
       'month', 'day', 'state'],
      dtype='object')

# Clean the Data

In [53]:
# clean the data
## step 1: convert to lowercase
data["clean_reason"] = (
    data["reason"]
    .str.lower()
)

data["clean_reason"].head()

0                 pgs 81 & 369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4    pages 28, 30 & 35 contain sexually explicit im...
Name: clean_reason, dtype: object

In [54]:
# clean the data
## step 2: remove punctuation
data["clean_reason"] = (
    data["clean_reason"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

data["clean_reason"].head()

0                  pgs 81  369 sexually explicit image
1                                    pg 422 nude child
2                       pg 294 sexually explicit image
3    page 9 of photo insert contains sexually expli...
4     pages 28 30  35 contain sexually explicit images
Name: clean_reason, dtype: object

In [55]:
# clean the data
## step 3: remove 'pgs' and numbers
data["clean_reason"] = (
    data["clean_reason"]
    .str.replace(r"\bpgs\b", "", regex = True) # remove pgs
    .str.replace(r"\bpg\b", "", regex = True) # remove pg
    .str.replace(r"\bpage\b", "", regex = True) # remove page
    .str.replace(r"\bpages\b", "", regex = True) # remove pages
    .str.replace(r"\d+", "", regex = True) # remove numbers
    .str.strip()
)

data["clean_reason"].head(100)

0                               sexually explicit image
1                                            nude child
2                               sexually explicit image
3     of photo insert contains sexually explicit images
4                      contain sexually explicit images
                            ...                        
95                                     sex with a minor
96                                incest brotherbrother
97                                                 rape
98                                                 rape
99                                                 rape
Name: clean_reason, Length: 100, dtype: object

In [56]:
# clean the data
## step 4: remove stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)
# import english stop words and make them a set called stopwords

In [57]:
data["clean_reason"] = data["clean_reason"].apply( # have column
    lambda x: " ".join(
        word for word in x.split()
        if word not in stopwords
    ) if isinstance(x,str) else "" # if data is not string, replace with string
)

data["clean_reason"].head(10)

0                           sexually explicit image
1                                        nude child
2                           sexually explicit image
3    photo insert contains sexually explicit images
4                  contain sexually explicit images
5                                    rape minor age
6                                        nude child
7                           sexually explicit image
8                  contains sexually explicit image
9                                        nude child
Name: clean_reason, dtype: object

In [62]:
# clean data: corpus title
## step 1: convert to lowercase
corpus["clean_title"] = (
    corpus["Title"]
    .str.lower()
)

corpus["clean_title"].head()

0                            the black witch
1                            shadow of night
2                  somewhere towards the end
3                              chain of iron
4    the lesbiana's guide to catholic school
Name: clean_title, dtype: object

In [63]:
# clean the data: corpus title
## step 2: remove punctuation
corpus["clean_title"] = (
    corpus["clean_title"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

corpus["clean_title"].head()

0                           the black witch
1                           shadow of night
2                 somewhere towards the end
3                             chain of iron
4    the lesbianas guide to catholic school
Name: clean_title, dtype: object

In [64]:
# clean the data: banned books titles
## step 1: convert to lowercase
data["clean_pub"] = (
    data["publication"]
    .str.lower()
)

# data["clean_pub"].head()

In [65]:
# clean the data: banned book titles
## step 2: remove punctuation
data["clean_pub"] = (
    data["clean_pub"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

# data["clean_pub"].head()

## Join datasets by title

In [66]:
merged = data.merge(
    corpus,
    left_on="clean_pub",
    right_on="clean_title",
    how="inner"
)

merged.head()

,publication,author,date,unit_reason,reason,year,month,day,state,clean_reason,clean_pub,Author,Title,Description,Genre,Banned,clean_title
0,A CLOCKWORK ORANGE,"KRAMER, PETER",2012-09-25,"PAGES: 21, 39, 43 & 54 CONTAIN SEXUALLY EXPLIC...","PGS 21, 39, 43 & 54 SEXUALLY EXPLICIT IMAGES",2012,9,25,tx,sexually explicit images,a clockwork orange,Anthony Burgess,A Clockwork Orange,A Clockwork Orange is Anthony Burgess’s most f...,"Fiction, Science Fiction, Dystopia, Horror, Li...",1,a clockwork orange
1,A CLOCKWORK ORANGE,"KUBRICK, STANLEY",2018-11-20,"AGES 4, 29, 45, 52, & 53 SEXUALLY EXPLICIT IMAGES","PGS 4, 29, 45, 52 & 53 SEXUALLY EXPLICIT IMAGES",2018,11,20,tx,sexually explicit images,a clockwork orange,Anthony Burgess,A Clockwork Orange,A Clockwork Orange is Anthony Burgess’s most f...,"Fiction, Science Fiction, Dystopia, Horror, Li...",1,a clockwork orange
2,A GIRL ON THE SHORE,"ASANO, INIO",2017-02-28,PGS264-269 CONTAIN SEXUALLY EXPLICIT IMAGES,PGS 264-269 SEXUALLY EXPLICIT IMAGES,2017,2,28,tx,sexually explicit images,a girl on the shore,"Inio, Asano",A Girl on the Shore,Longing for Something Bigger From the Eisner ...,"nullManga, Graphic Novels, Comics, Romance, Fi...",1,a girl on the shore
3,A GIRL ON THE SHORE,"ASANO, INIO",2017-02-28,PGS264-269 CONTAIN SEXUALLY EXPLICIT IMAGES,PGS 264-269 SEXUALLY EXPLICIT IMAGES,2017,2,28,tx,sexually explicit images,a girl on the shore,Asano Inio,A Girl on the Shore,Longing for Something Bigger From the Eisner ...,"nullManga, Graphic Novels, Comics, Romance, Fi...",1,a girl on the shore
4,A QUIET BELIEF IN ANGELS,"ELLORY, R. J.",2011-01-04,PAGES 202 & 203 RAPE PAGE 202 AGE,PAGES: 202 & 203 RAPE PAGE: 202 AGE,2011,1,4,tx,rape age,a quiet belief in angels,R.J. Ellory,A Quiet Belief in Angels,Joseph Vaughan's life has been dogged by trage...,"Crime, Fiction, Mystery, Thriller, Historical ...",0,a quiet belief in angels


In [67]:
merged.shape

(314, 17)

### Are there differences between unit_reason and reason?

In [24]:
## step 1: convert to lowercase
data["clean_unit_reason"] = (
    data["unit_reason"]
    .str.lower()
)

# data["clean_unit_reason"].head()

In [25]:
## step 2: remove punctuation
data["clean_unit_reason"] = (
    data["clean_unit_reason"]
    .str.translate(
        str.maketrans("", "", string.punctuation)
    )
)

# data["clean_unit_reason"].head()

In [26]:
## step 3: remove 'pgs' and numbers
data["clean_unit_reason"] = (
    data["clean_unit_reason"]
    .str.replace(r"\bpgs\b", "", regex = True) # remove pgs
    .str.replace(r"\bpg\b", "", regex = True) # remove pg
    .str.replace(r"\bpage\b", "", regex = True) # remove page
    .str.replace(r"\bpages\b", "", regex = True) # remove pages
    .str.replace(r"\d+", "", regex = True) # remove numbers
    .str.strip()
)

# data["clean_unit_reason"].head(100)

In [18]:
## step 4: remove stopwords
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stopwords = set(ENGLISH_STOP_WORDS)

In [27]:
data["clean_unit_reason"] = data["clean_unit_reason"].apply( # have column
    lambda x: " ".join(
        word for word in x.split()
        if word not in stopwords
    ) if isinstance(x,str) else "" # if data is not string, replace with string
)

# data["clean_unit_reason"].head(10)

## Queries

In [20]:
# find most common words in reason column
all_words = " ".join(data["clean_reason"].dropna()).lower().split()

Counter(all_words).most_common(50)

[('sexually', 3906),
 ('explicit', 3868),
 ('images', 3290),
 ('rape', 1355),
 ('contain', 1116),
 ('contains', 852),
 ('child', 783),
 ('image', 665),
 ('nude', 597),
 ('sei', 471),
 ('techniques', 469),
 ('sex', 395),
 ('age', 364),
 ('fighting', 352),
 ('entire', 344),
 ('book', 336),
 ('wminor', 278),
 ('incest', 240),
 ('minor', 229),
 ('escape', 206),
 ('offdef', 200),
 ('ind', 194),
 ('facilitate', 186),
 ('mfg', 176),
 ('detailed', 168),
 ('indecency', 166),
 ('information', 166),
 ('wchild', 160),
 ('photo', 143),
 ('texas', 143),
 ('graphic', 134),
 ('depiction', 131),
 ('wpn', 125),
 ('security', 124),
 ('manipulation', 111),
 ('maps', 104),
 ('cover', 102),
 ('manufacture', 96),
 ('insert', 92),
 ('brk', 86),
 ('ftrdtr', 85),
 ('offensivedefensive', 85),
 ('used', 82),
 ('concerns', 79),
 ('drugs', 73),
 ('weapon', 71),
 ('bst', 68),
 ('brosis', 62),
 ('ages', 62),
 ('chl', 59)]

In [21]:
# find most common words in unit_reason column
all_words = " ".join(data["clean_unit_reason"].dropna()).lower().split()

Counter(all_words).most_common(25)

[('sexually', 3825),
 ('explicit', 3796),
 ('images', 3252),
 ('contains', 1610),
 ('contain', 1571),
 ('rape', 1461),
 ('ages', 1081),
 ('age', 896),
 ('child', 680),
 ('book', 584),
 ('image', 581),
 ('minor', 550),
 ('sex', 545),
 ('sei', 504),
 ('f', 493),
 ('nude', 459),
 ('reason', 449),
 ('entire', 412),
 ('techniques', 391),
 ('g', 274),
 ('fighting', 273),
 ('incest', 253),
 ('gs', 233),
 ('information', 190),
 ('graphic', 161)]

### Let's use N-grams

In [22]:
from nltk import ngrams

In [23]:
# all_words is from the reason column
bigrams = list(
    ngrams(all_words, 2)
)

Counter(bigrams).most_common(25)

[(('sexually', 'explicit'), 3681),
 (('explicit', 'images'), 3103),
 (('contain', 'sexually'), 898),
 (('images', 'sexually'), 783),
 (('contains', 'sexually'), 641),
 (('explicit', 'image'), 551),
 (('nude', 'child'), 403),
 (('entire', 'book'), 398),
 (('ages', 'contain'), 374),
 (('images', 'ages'), 364),
 (('reason', 'f'), 364),
 (('images', 'contain'), 307),
 (('sex', 'minor'), 286),
 (('book', 'contains'), 242),
 (('fighting', 'techniques'), 239),
 (('age', 'contains'), 236),
 (('minor', 'age'), 232),
 (('ages', 'sexually'), 229),
 (('rape', 'minor'), 204),
 (('images', 'contains'), 190),
 (('rape', 'rape'), 185),
 (('contains', 'rape'), 177),
 (('images', 'rape'), 177),
 (('ages', 'contains'), 160),
 (('rape', 'sexually'), 150)]

# Conclusions
## Data Cleaning
### After cleaning the data and filtering the most common informative words, recurring reasons for banning a book in Texas state prisons are as follows: 
### 1. sexually explicit images
### 2. nude child & sex wminor
### 3. rape
### 4. incest
### 5. fighting techniques
### 6. ????

In [24]:
data.shape

(9396, 11)

In [25]:
# sum common reasons
## sexually explicit images or sei
data["clean_reason"].str.contains(
    r"sei|sexually explicit images",
    case=False,
    na=False
).sum()

np.int64(3693)

## Filtered Copy of Data

In [28]:
filtered = data.copy()

#3693
filtered = filtered.loc[
    ~filtered["clean_reason"].str.contains(
        r"sei|sexually explicit images|rape",
        case=False,
        na=False
    )
]

#1346
#filtered["clean_reason"].str.contains(
   # "rape",
   # case=False,
   # na=False
#).sum()

np.int64(1346)

In [87]:
## nude child
data["clean_reason"].str.contains(
    r"nude child|nud chld",
    case=False,
    na=False
).sum()

np.int64(591)

In [86]:
## offdef or offensivedefensive
data["clean_reason"].str.contains(
    r"fighting techniques|offdef|offensivedefensive",
    case=False,
    na=False
).sum()

np.int64(363)

In [80]:
## sex wminor or sex minor
data["clean_reason"].str.contains(
    r"sex wminor|sex minor",
    case=False,
    na=False
).sum()

np.int64(354)

In [85]:
## ind wchild or indecency child
data["clean_reason"].str.contains(
    r"ind wchild|indecency child",
    case=False,
    na=False
).sum()

np.int64(295)

In [65]:
## incest
data["clean_reason"].str.contains("incest", case=False, na=False).sum()

np.int64(229)

In [74]:
## escape
data["clean_reason"].str.contains("escape", case=False, na=False).sum()

np.int64(206)

In [84]:
## security
data["clean_reason"].str.contains("security", case=False, na=False).sum()

np.int64(120)

## Output Queries

In [36]:
data[data["reason"].str.contains("security concern", case=False, na=False)]

,publication,author,date,reason,year,month,day,state
102,A MESSAGE TO EVERY PRISONER IN AMERICA,"JONES, BRYACE",2022-08-30,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,8,30,tx
111,A PANTHER IS A BLACK CAT,"MAJOR, REGINALD",2015-01-14,ENTIRE BOOK CONTAINS INFORMATION THAT COULD CA...,2015,1,14,tx
316,ALL IN ONE COMPTIA NETWORK + EXAM N10-005,"MEYERS, MIKE",2022-03-03,ENTIRE PUBLICATION CONTAINS SECURITY CONCERNS,2022,3,3,tx
328,ALL-IN-ONE COMPTIA NETWORK + CERTIFICATION EXAM,"MEYERS, MIKE",2022-03-28,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,3,28,tx
406,AN AUTHENTIC GUIDE TO KALI LINUX,"BHOOPLI, SAMARAT",2020-02-03,"PAGES 10, 11, 50, 61 & 62 CONTAIN INFORMATION ...",2020,2,3,tx
...,...,...,...,...,...,...,...,...
7840,SUMMARY THE LAWS OF HUMAN NATURE,"GREENE, ROBERT",2020-01-03,ENTIRE BOOK CONTAINS MANIPULATION TECHNIQUES W...,2020,1,3,tx
8860,WHEN HELL BECOMES YOUR HOME,"COGNITO, N.",2022-03-31,ENTIRE BOOK CONTAINS SECURITY CONCERNS,2022,3,31,tx
8969,WINDOWS KERNAL PROGRAMMING,"YOSIFOVICH, PAVEL",2021-12-15,ENTIRE BOOK CONTAINS SECURITY CONCERNS,2021,12,15,tx
8990,WIRESHARK FOR SECURITY PROFESSIONALS,"BULLOCK, JESSEY",2022-07-28,ENTIRE BOOK CONTAINS INFORMATION DEEMED A SECU...,2022,7,28,tx


## Join Google Books Data w/ Banned Books Data